In [ ]:
import re
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

# =========================
# 結果ファイルのパス（results/ ディレクトリ配下）
# =========================
MD = [
    "../results/multi/scail-FEP/10/mtl_eval_summary523.txt",
    "../results/multi/scail-FEP/15/mtl_eval_summary523.txt",
    "../results/multi/scail-FEP/23/mtl_eval_summary523.txt",
    "../results/multi/scail-FEP/35/mtl_eval_summary523.txt",
    "../results/multi/scail-FEP/53/mtl_eval_summary523.txt",
    "../results/multi/scail-FEP/80/mtl_eval_summary523.txt",
    "../results/multi/scail-FEP/121/mtl_eval_summary523.txt",
    "../results/multi/scail-FEP/184/mtl_eval_summary523.txt",
    "../results/multi/scail-FEP/279/mtl_eval_summary523.txt",
    "../results/multi/scail-FEP/435_409/mtl_eval_summary523.txt",
]

rosetta = [
    "../results/multi/scail-rosetta/10/mtl_eval_summary523.txt",
    "../results/multi/scail-rosetta/10/mtl_eval_summary523.txt",
    "../results/multi/scail-rosetta/23/mtl_eval_summary523.txt",
    "../results/multi/scail-rosetta/35/mtl_eval_summary523.txt",
    "../results/multi/scail-rosetta/53/mtl_eval_summary523.txt",
    "../results/multi/scail-rosetta/80/mtl_eval_summary523.txt",
    "../results/multi/scail-rosetta/121/mtl_eval_summary523.txt",
    "../results/multi/scail-rosetta/184/mtl_eval_summary523.txt",
    "../results/multi/scail-rosetta/279/mtl_eval_summary523.txt",
    "../results/multi/scail-rosetta/435_409/mtl_eval_summary523.txt",
    
]

MPNN = [
    "../results/multi/scail-thermoMPNN/10/mtl_eval_summary523.txt",
    "../results/multi/scail-thermoMPNN/15/mtl_eval_summary523.txt",
    "../results/multi/scail-thermoMPNN/23/mtl_eval_summary523.txt",
    "../results/multi/scail-thermoMPNN/35/mtl_eval_summary523.txt",
    "../results/multi/scail-thermoMPNN/53/mtl_eval_summary523.txt",
    "../results/multi/scail-thermoMPNN/80/mtl_eval_summary523.txt",
    "../results/multi/scail-thermoMPNN/121/mtl_eval_summary523.txt",
    "../results/multi/scail-thermoMPNN/184/mtl_eval_summary523.txt",
    "../results/multi/scail-thermoMPNN/279/mtl_eval_summary523.txt",
    "../results/multi/scail-thermoMPNN/435_409/mtl_eval_summary523.txt",
    
]

# === foldX 追加 ===
foldX = [
    "../results/multi/scail-FoldX/10/mtl_eval_summary523.txt",
    "../results/multi/scail-FoldX/15/mtl_eval_summary523.txt",
    "../results/multi/scail-FoldX/23/mtl_eval_summary523.txt",
    "../results/multi/scail-FoldX/35/mtl_eval_summary523.txt",
    "../results/multi/scail-FoldX/53/mtl_eval_summary523.txt",
    "../results/multi/scail-FoldX/80/mtl_eval_summary523.txt",
    "../results/multi/scail-FoldX/121/mtl_eval_summary523.txt",
    "../results/multi/scail-FoldX/184/mtl_eval_summary523.txt",
    "../results/multi/scail-FoldX/279/mtl_eval_summary523.txt",
    "../results/multi/scail-FoldX/435_409/mtl_eval_summary523.txt",
    
]

datasize = np.array([10*2, 15*2, 23*2, 35*2, 53*2, 80*2, 121*2, 184*2, 279*2, 423*2], dtype=float)

TARGET_METRIC = "MAE"
baseline = 7.750233

LINE_RE = re.compile(
    r"^\s*(?P<metric>\w+)\s+mean\s*=\s*(?P<mean>[-+0-9.eE]+)\s+"
    r"90%\s*CI\s*=\s*\[\s*(?P<lo>[-+0-9.eE]+)\s*,\s*(?P<hi>[-+0-9.eE]+)\s*\]\s*$"
)

def read_metric(paths, metric="MAE"):
    means, cis = [], []
    for p in paths:
        mean = lo = hi = None
        with open(p, "r", encoding="utf-8") as f:
            for line in f:
                m = LINE_RE.match(line)
                if m and m.group("metric") == metric:
                    mean = float(m.group("mean"))
                    lo   = float(m.group("lo"))
                    hi   = float(m.group("hi"))
                    break
        if mean is None:
            raise RuntimeError(f"{p} に {metric} の mean/CI 行が見つかりません")
        means.append(mean)
        cis.append([lo, hi])
    return np.array(means, dtype=float), np.array(cis, dtype=float)

def power_law(n, a, b, c):
    return a * n**b + c

def fit_powerlaw(datasize, means):
    x_raw = np.array(datasize, dtype=float)
    y = np.array(means, dtype=float)
    x = x_raw / 1000.0

    c0 = float(np.min(y) - 0.02)
    a0 = float(np.max(y) - c0)
    b0 = -0.2

    bounds = ((1e-6, -3.0, 6.0), (5.0, -1e-3, 7.0))

    try:
        popt, _ = curve_fit(
            power_law, x, y,
            p0=[a0, b0, c0],
            bounds=bounds,
            maxfev=20000
        )
        a, b, c = popt
        used_backup = False
    except Exception:
        c_grid = np.linspace(np.min(y) - 0.3, np.min(y) - 1e-3, 600)
        logx = np.log(x)
        best_sse, best_params = np.inf, None
        for c_try in c_grid:
            diff = y - c_try
            if np.any(diff <= 0):
                continue
            logy = np.log(diff)
            X = np.vstack([np.ones_like(logx), logx]).T
            beta, *_ = np.linalg.lstsq(X, logy, rcond=None)
            A, b_try = beta
            a_try = np.exp(A)
            y_pred = a_try * x**b_try + c_try
            sse = np.sum((y - y_pred) ** 2)
            if sse < best_sse:
                best_sse = sse
                best_params = (a_try, b_try, c_try)
        if best_params is None:
            raise RuntimeError("backup フィットも失敗しました")
        a, b, c = best_params
        used_backup = True

    return a, b, c, used_backup

assert len(MD) == len(datasize)
assert len(rosetta) == len(datasize)
assert len(MPNN) == len(datasize)
assert len(foldX) == len(datasize)

means1, ci1 = read_metric(MD)
means2, ci2 = read_metric(rosetta)
means3, ci3 = read_metric(MPNN)
means4, ci4 = read_metric(foldX)

a1, b1, c1, backup1 = fit_powerlaw(datasize, means1)
a2, b2, c2, backup2 = fit_powerlaw(datasize, means2)
a3, b3, c3, backup3 = fit_powerlaw(datasize, means3)
a4, b4, c4, backup4 = fit_powerlaw(datasize, means4)

plt.plot(datasize, means1, 'o-', color='blue', label="Tm + ΔΔG(FEP)")
plt.fill_between(datasize, ci1[:,0], ci1[:,1], color='lightgray', alpha=0.4)

plt.plot(datasize, means2, 's--', color='green', label="Tm + ΔΔG(rosetta)")
plt.fill_between(datasize, ci2[:,0], ci2[:,1], color='lightgray', alpha=0.4)

plt.plot(datasize, means3, 'd-.', color='orange', label="Tm + ΔΔG(MPNN)")
plt.fill_between(datasize, ci3[:,0], ci3[:,1], color='lightgray', alpha=0.4)

plt.plot(datasize, means4, 'x-', color='purple', label="Tm + ΔΔG(foldX)")
plt.fill_between(datasize, ci4[:,0], ci4[:,1], color='lightgray', alpha=0.4)

plt.axhline(y=baseline, color='darkred', linestyle='--', label="Single-task")

x_fit = np.logspace(np.log10(datasize.min()), np.log10(datasize.max()), 200)

plt.plot(x_fit, a1*(x_fit/1000.0)**b1 + c1, color='blue', linestyle=':')
plt.plot(x_fit, a2*(x_fit/1000.0)**b2 + c2, color='green', linestyle=':')
plt.plot(x_fit, a3*(x_fit/1000.0)**b3 + c3, color='orange', linestyle=':')
plt.plot(x_fit, a4*(x_fit/1000.0)**b4 + c4, color='purple', linestyle=':')

plt.xscale("log")
plt.xlabel("Number of simulation samples")
plt.ylabel(TARGET_METRIC)
plt.legend()
plt.grid(True, linestyle="--", alpha=0.7)
plt.show()

print("MD     :", a1, b1, c1)
print("rosetta:", a2, b2, c2)
print("MPNN   :", a3, b3, c3)
print("foldX  :", a4, b4, c4)